# 🚀 ZAIC 2026 DATAMATE — THÍ NGHIỆM SLM TEXT-TO-SQL TRÊN COLAB PRO A100

> **Track 2: DataMate — Phase 3: Hybrid Fast-Slow Architecture**  
> **Mục tiêu:** Chạy thử nghiệm mô hình nền `Qwen/Qwen2.5-Coder-7B-Instruct` (hoặc `14B`) trên GPU A100 để mở khóa ~796 câu hỏi long-tail mà Fast Path đang từ chối an toàn (`abstain`).

---

### 1. Kiểm tra Cấu hình Phần cứng GPU A100

In [ ]:
!nvidia-smi

### 2. Cài đặt Môi trường vLLM & Thư viện Cần thiết

In [ ]:
# Cài đặt vLLM engine và các gói xử lý dữ liệu
!pip install -q vllm duckdb pandas openpyxl python-calamine rapidfuzz

### 3. Tùy chọn A: Khởi động vLLM OpenAI-Compatible Server & Cloudflare Tunnel
> Chạy cell này nếu bạn muốn Antigravity Agent ở máy nhà gọi trực tiếp Colab A100 qua API endpoint `*.trycloudflare.com`.

In [ ]:
import subprocess, time, re, sys

# 1. Tải cloudflared nếu chưa có
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# 2. Khởi động vLLM Server chạy ngầm với Qwen2.5-Coder-7B-Instruct
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
print(f"📦 Đang khởi động vLLM server cho {MODEL_ID}...")
vllm_cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--port", "8000",
    "--max-model-len", "8192",
    "--gpu-memory-utilization", "0.90",
    "--dtype", "bfloat16",
    "--trust-remote-code"
]
vllm_proc = subprocess.Popen(vllm_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Đợi 20s cho vLLM nạp model vào VRAM
print("⏳ Đang nạp model vào VRAM A100 (khoảng 30-45 giây)...", flush=True)
time.sleep(35)

# 3. Mở đường hầm Cloudflare Tunnel cho port 8000
print("🌐 Đang thiết lập Cloudflare Tunnel...", flush=True)
cf_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"], stderr=subprocess.PIPE, text=True)

tunnel_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = cf_proc.stderr.readline()
    match = re.search(r"https://([a-zA-Z0-9-]+\.trycloudflare\.com)", line)
    if match:
        tunnel_url = match.group(1)
        break
    time.sleep(0.1)

if tunnel_url:
    print("\n" + "="*70)
    print("🎉 API ENDPOINT VLLM SẴN SÀNG!")
    print(f"👉 Base URL: https://{tunnel_url}/v1")
    print("="*70 + "\n")
    print("Bạn có thể cấu hình ở máy local bằng lệnh:")
    print(f"export LLM_BASE_URL='https://{tunnel_url}/v1'")
    print("uv run python scripts/benchmark_hybrid.py --enable-slow-path --limit 50")
else:
    print("❌ Chưa bắt được đường hầm Cloudflare, hãy kiểm tra lại kết nối mạng.")

### 4. Tùy chọn B: Chạy Đánh giá Trực tiếp trong Colab (Offline Inference)
> Sử dụng vLLM `LLM` class trực tiếp để sinh câu lệnh SQL hàng loạt và đo lường Penalized Accuracy ngay trong notebook.

In [ ]:
from vllm import LLM, SamplingParams

# Khởi tạo model trực tiếp
llm = LLM(model="Qwen/Qwen2.5-Coder-7B-Instruct", tensor_parallel_size=1, max_model_len=8192, gpu_memory_utilization=0.90)

# Cấu hình tham số sinh theo Rule 4.4 bất biến (temperature=0.0, seed=42)
sampling_params = SamplingParams(temperature=0.0, seed=42, max_tokens=1024)
print("✅ Mô hình Qwen2.5-Coder-7B đã sẵn sàng cho suy luận trực tiếp trên A100!")